In [4]:
from pathlib import Path
import json

# Build a CLEAN notebook: only the dataset-building workflow is inside it.
cells_source = [
r'''# Revenue Counterfactual Engine — Dataset Builder
# Clean version: this notebook builds the dataset; it does NOT create another notebook.

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("REVENUE COUNTERFACTUAL ENGINE — DATASET BUILDER")
print("=" * 60)
print("This notebook combines the three source datasets as separate")
print("evidence layers and creates a purpose-built benchmark dataset.")
print()
''',

r'''# 1. Enter the three CSV paths

path_1 = input("1) E-commerce Customer Behavior CSV path: ").strip().strip('"').strip("'")
path_2 = input("2) Sales Transaction CSV path: ").strip().strip('"').strip("'")
path_3 = input("3) Retail Fraud Detection CSV path: ").strip().strip('"').strip("'")

for i, p in enumerate([path_1, path_2, path_3], 1):
    if not Path(p).is_file():
        raise FileNotFoundError(
            f"Dataset {i} was not found:\n{p}\n\n"
            "Check the path and run this cell again."
        )

print("\nAll three files were found.")
''',

r'''# 2. Load and normalize

def load_csv(path):
    attempts = [
        {"low_memory": False},
        {"low_memory": False, "encoding": "utf-8"},
        {"low_memory": False, "encoding": "latin1"},
    ]
    last_error = None
    for kwargs in attempts:
        try:
            return pd.read_csv(path, **kwargs)
        except Exception as e:
            last_error = e
    raise last_error

def normalize_columns(df):
    out = df.copy()
    out.columns = [
        re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
        for c in out.columns
    ]
    return out

df_behavior = normalize_columns(load_csv(path_1))
df_sales = normalize_columns(load_csv(path_2))
df_fraud = normalize_columns(load_csv(path_3))

print(f"Dataset 1: {df_behavior.shape[0]:,} rows × {df_behavior.shape[1]} columns")
print(f"Dataset 2: {df_sales.shape[0]:,} rows × {df_sales.shape[1]} columns")
print(f"Dataset 3: {df_fraud.shape[0]:,} rows × {df_fraud.shape[1]} columns")

print("\nDataset 1 columns:")
print(df_behavior.columns.tolist())
print("\nDataset 2 columns:")
print(df_sales.columns.tolist())
print("\nDataset 3 columns:")
print(df_fraud.columns.tolist())
''',

r'''# 3. Robust column helpers

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def numeric_col(df, candidates, default=np.nan):
    c = first_existing(df, candidates)
    if c is None:
        return pd.Series(default, index=df.index, dtype="float64")
    return pd.to_numeric(df[c], errors="coerce")

def string_col(df, candidates, default="unknown"):
    c = first_existing(df, candidates)
    if c is None:
        return pd.Series(default, index=df.index, dtype="string")
    return df[c].astype("string").fillna(default)

def bool_col(df, candidates, default=False):
    c = first_existing(df, candidates)
    if c is None:
        return pd.Series(default, index=df.index, dtype="boolean")

    s = df[c]
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(default).astype("boolean")

    return (
        s.astype("string").str.strip().str.lower()
        .map({
            "true": True, "false": False,
            "1": True, "0": False,
            "yes": True, "no": False,
            "y": True, "n": False
        })
        .fillna(default)
        .astype("boolean")
    )

def safe_clip(series, low=0, high=None):
    s = pd.to_numeric(series, errors="coerce")
    if high is None:
        return s.clip(lower=low)
    return s.clip(lower=low, upper=high)
''',

r'''# 4. Dataset 1 — customer/session behavior features

b = df_behavior.copy()

behavior = pd.DataFrame(index=b.index)

behavior["source_behavior_customer_id"] = string_col(
    b, ["customer_id", "customerid"]
)
behavior["behavior_date"] = pd.to_datetime(
    string_col(b, ["date", "order_date", "transaction_date"], default=""),
    errors="coerce"
)

behavior["customer_age"] = numeric_col(b, ["age"])
behavior["customer_gender"] = string_col(b, ["gender"])
behavior["customer_city"] = string_col(b, ["city"])

behavior["product_category"] = string_col(
    b, ["product_category", "category"]
)
behavior["unit_price"] = numeric_col(b, ["unit_price", "price"])
behavior["quantity"] = numeric_col(b, ["quantity"])
behavior["discount_amount"] = numeric_col(
    b, ["discount_amount", "discount"]
)
behavior["behavior_total_amount"] = numeric_col(
    b, ["total_amount", "amount", "sales"]
)

behavior["payment_method"] = string_col(
    b, ["payment_method", "payment_type"]
)
behavior["device_type"] = string_col(
    b, ["device_type", "device"]
)
behavior["session_duration_minutes"] = numeric_col(
    b, ["session_duration_minutes", "session_duration"]
)
behavior["pages_viewed"] = numeric_col(
    b, ["pages_viewed", "pages"]
)
behavior["is_returning_customer"] = bool_col(
    b, ["is_returning_customer", "returning_customer"]
)
behavior["delivery_time_days"] = numeric_col(
    b, ["delivery_time_days", "delivery_days"]
)
behavior["customer_rating"] = numeric_col(
    b, ["customer_rating", "rating"]
)

behavior["discount_rate"] = np.where(
    behavior["behavior_total_amount"].fillna(0) +
    behavior["discount_amount"].fillna(0) > 0,
    behavior["discount_amount"].fillna(0) /
    (behavior["behavior_total_amount"].fillna(0) +
     behavior["discount_amount"].fillna(0)),
    0
)

print("Useful behavior features extracted:", len(behavior.columns))
''',

r'''# 5. Dataset 2 — long-term purchase history

s = df_sales.copy()

customer_col = first_existing(
    s, ["customer", "customer_id", "customerid", "customer_name"]
)
date_col = first_existing(
    s, ["date", "invoice_date", "transaction_date", "order_date"]
)
amount_col = first_existing(
    s, ["amount", "total_amount", "sales", "revenue"]
)
price_col = first_existing(s, ["price", "unit_price"])
quantity_col = first_existing(s, ["quantity", "qty", "units"])
product_col = first_existing(
    s, ["product_name", "product", "description", "product_id"]
)
country_col = first_existing(s, ["country", "customer_country"])

sales = pd.DataFrame(index=s.index)

sales["customer_key"] = (
    s[customer_col].astype("string").fillna("UNKNOWN_CUSTOMER")
    if customer_col
    else pd.Series("UNKNOWN_CUSTOMER", index=s.index, dtype="string")
)

sales["date"] = (
    pd.to_datetime(s[date_col], errors="coerce")
    if date_col
    else pd.Series(pd.NaT, index=s.index)
)

if amount_col:
    sales["amount"] = pd.to_numeric(s[amount_col], errors="coerce")
else:
    sales["amount"] = np.nan

sales["quantity"] = (
    pd.to_numeric(s[quantity_col], errors="coerce")
    if quantity_col
    else 1.0
)

if price_col:
    price = pd.to_numeric(s[price_col], errors="coerce")
    calculated_amount = price * sales["quantity"].fillna(1)
    sales["amount"] = sales["amount"].fillna(calculated_amount)

sales["product"] = (
    s[product_col].astype("string").fillna("unknown_product")
    if product_col
    else pd.Series("unknown_product", index=s.index, dtype="string")
)

sales["country"] = (
    s[country_col].astype("string").fillna("unknown")
    if country_col
    else pd.Series("unknown", index=s.index, dtype="string")
)

sales = sales.sort_values(["customer_key", "date"])

reference_date = sales["date"].max()
if pd.isna(reference_date):
    reference_date = pd.Timestamp.today().normalize()

customer_history = sales.groupby("customer_key", dropna=False).agg(
    historical_transactions=("customer_key", "size"),
    historical_total_spend=("amount", "sum"),
    historical_average_order_value=("amount", "mean"),
    historical_max_order_value=("amount", "max"),
    historical_total_quantity=("quantity", "sum"),
    first_purchase_date=("date", "min"),
    last_purchase_date=("date", "max"),
    distinct_products=("product", "nunique"),
    distinct_countries=("country", "nunique")
).reset_index()

customer_history["days_since_last_purchase"] = (
    reference_date - customer_history["last_purchase_date"]
).dt.days

customer_history["customer_lifetime_days"] = (
    customer_history["last_purchase_date"] -
    customer_history["first_purchase_date"]
).dt.days.clip(lower=0)

customer_history["purchase_frequency_per_30d"] = np.where(
    customer_history["customer_lifetime_days"] > 0,
    customer_history["historical_transactions"] /
    (customer_history["customer_lifetime_days"] / 30),
    customer_history["historical_transactions"]
)

top_product = (
    sales.groupby(["customer_key", "product"])
    .size()
    .reset_index(name="purchase_count")
    .sort_values(
        ["customer_key", "purchase_count"],
        ascending=[True, False]
    )
    .drop_duplicates("customer_key")
    .rename(columns={"product": "favorite_product"})
    [["customer_key", "favorite_product"]]
)

customer_history = customer_history.merge(
    top_product, on="customer_key", how="left"
)

try:
    customer_history["customer_value_segment"] = pd.qcut(
        customer_history["historical_total_spend"].rank(method="first"),
        q=4,
        labels=["low", "medium", "high", "very_high"]
    ).astype("string")
except Exception:
    customer_history["customer_value_segment"] = "unknown"

print("Customer profiles created:", len(customer_history))
''',

r'''# 6. Dataset 3 — transaction risk signals
# Fraud labels are NOT used as agent inputs.

f = df_fraud.copy()

risk = pd.DataFrame(index=f.index)

risk["risk_transaction_id"] = string_col(
    f, ["transaction_id", "transactionid", "txn_id", "id"]
)

risk["risk_amount"] = numeric_col(
    f, ["amount", "transaction_amount", "total_amount"]
)

risk["risk_payment_method"] = string_col(
    f, ["payment_method", "payment_type", "payment_method_type"]
)

risk["risk_device_type"] = string_col(
    f, ["device_type", "device"]
)

risk["failed_transactions_24h"] = numeric_col(
    f,
    ["failed_transactions_24h",
     "failed_transaction_count_24h",
     "failed_transactions_last_24h"],
    default=0
)

risk["transaction_frequency_24h"] = numeric_col(
    f,
    ["transaction_frequency_24h",
     "transaction_frequency_last_24h",
     "transactions_24h"],
    default=0
)

risk["unusual_amount_flag"] = bool_col(
    f, ["unusual_amount_flag", "unusual_amount"]
)
risk["unusual_location_flag"] = bool_col(
    f, ["unusual_location_flag", "unusual_location"]
)
risk["multiple_transactions_short_time"] = bool_col(
    f,
    ["multiple_transactions_short_time",
     "multiple_transactions_short_period"]
)
risk["high_risk_device_flag"] = bool_col(
    f, ["high_risk_device_flag", "high_risk_device"]
)
risk["velocity_flag"] = bool_col(
    f, ["velocity_flag", "high_velocity_flag"]
)
risk["previous_fraud_flag"] = bool_col(
    f, ["previous_fraud_flag", "previous_fraud"]
)

risk["risk_score"] = numeric_col(
    f, ["fraud_risk", "risk_score", "fraud_score"]
)

fraud_label_col = first_existing(
    f, ["fraud_flag", "is_fraud", "fraud", "fraud_label"]
)

if fraud_label_col:
    risk["evaluation_fraud_label"] = bool_col(
        f, [fraud_label_col]
    )
else:
    risk["evaluation_fraud_label"] = pd.Series(
        pd.NA, index=f.index, dtype="boolean"
    )

risk["risk_signal_count"] = (
    risk["unusual_amount_flag"].astype(int)
    + risk["unusual_location_flag"].astype(int)
    + risk["multiple_transactions_short_time"].astype(int)
    + risk["high_risk_device_flag"].astype(int)
    + risk["velocity_flag"].astype(int)
    + risk["previous_fraud_flag"].astype(int)
    + (risk["failed_transactions_24h"].fillna(0) >= 3).astype(int)
)

risk["automated_recovery_risk"] = pd.cut(
    risk["risk_signal_count"],
    bins=[-1, 1, 3, 100],
    labels=["low", "medium", "high"]
).astype("string")

print("Risk profiles created:", len(risk))
''',

r'''# 7. Create the purpose-built synthetic revenue-event benchmark

# This is intentionally synthetic because the three source datasets
# do not contain the complete recovery lifecycle.
N_EVENTS = 10000
rng = np.random.default_rng(42)

behavior_sample = behavior.sample(
    N_EVENTS, replace=True, random_state=42
).reset_index(drop=True)

history_sample = customer_history.sample(
    N_EVENTS, replace=True, random_state=43
).reset_index(drop=True)

risk_sample = risk.sample(
    N_EVENTS, replace=True, random_state=44
).reset_index(drop=True)

events = pd.DataFrame()
events["case_id"] = [
    f"CASE_{i:06d}" for i in range(1, N_EVENTS + 1)
]

# -------- customer behavior --------
for col in [
    "customer_age", "customer_gender", "customer_city",
    "payment_method", "device_type",
    "session_duration_minutes", "pages_viewed",
    "is_returning_customer"
]:
    events[col] = behavior_sample[col]

# -------- long-term history --------
for col in [
    "historical_transactions",
    "historical_total_spend",
    "historical_average_order_value",
    "historical_max_order_value",
    "purchase_frequency_per_30d",
    "days_since_last_purchase",
    "distinct_products",
    "customer_value_segment",
    "favorite_product"
]:
    events[col] = history_sample[col]

# -------- risk --------
for col in [
    "failed_transactions_24h",
    "transaction_frequency_24h",
    "unusual_amount_flag",
    "unusual_location_flag",
    "multiple_transactions_short_time",
    "high_risk_device_flag",
    "velocity_flag",
    "previous_fraud_flag",
    "risk_signal_count",
    "automated_recovery_risk"
]:
    events[col] = risk_sample[col]

# Transaction amount: prefer observed e-commerce amount,
# then historical AOV, then risk amount.
events["transaction_amount"] = (
    behavior_sample["behavior_total_amount"]
    .fillna(history_sample["historical_average_order_value"])
    .fillna(risk_sample["risk_amount"])
)

fallback_amount = events["transaction_amount"].median()
if pd.isna(fallback_amount) or fallback_amount <= 0:
    fallback_amount = 500.0

events["transaction_amount"] = (
    events["transaction_amount"]
    .where(events["transaction_amount"] > 0, fallback_amount)
    .round(2)
)

# -------- event types --------
event_types = [
    "payment_failure",
    "checkout_abandonment",
    "subscription_failure",
    "invoice_overdue",
    "repeated_payment_failure"
]

events["revenue_event"] = rng.choice(
    event_types,
    size=N_EVENTS,
    p=[0.35, 0.25, 0.15, 0.15, 0.10]
)

failure_map = {
    "payment_failure": [
        "bank_timeout", "temporary_bank_error",
        "payment_method_failure", "insufficient_funds"
    ],
    "checkout_abandonment": [
        "checkout_abandoned", "payment_page_abandoned",
        "customer_inactivity"
    ],
    "subscription_failure": [
        "subscription_payment_failed",
        "expired_payment_method",
        "recurring_payment_failure"
    ],
    "invoice_overdue": [
        "invoice_overdue",
        "payment_not_received",
        "customer_delay"
    ],
    "repeated_payment_failure": [
        "repeated_payment_failure",
        "multiple_bank_failures",
        "repeated_declines"
    ]
}

events["failure_reason"] = [
    rng.choice(failure_map[t]) for t in events["revenue_event"]
]

# Retry history
events["retry_count"] = 0

repeated = events["revenue_event"].eq("repeated_payment_failure")
events.loc[repeated, "retry_count"] = rng.integers(
    2, 4, repeated.sum()
)

payment = events["revenue_event"].eq("payment_failure")
events.loc[payment, "retry_count"] = rng.choice(
    [0, 1], payment.sum(), p=[0.75, 0.25]
)
''',

r'''# 8. Build safety/risk eligibility

events["high_risk_case"] = (
    events["automated_recovery_risk"].eq("high")
    | events["previous_fraud_flag"].fillna(False)
    | events["velocity_flag"].fillna(False)
)

events["recovery_eligible"] = ~events["high_risk_case"]

# Customer strength: a synthetic propensity feature derived from
# observed historical/customer behavior.
customer_strength = (
    0.30
    + 0.10 * events["is_returning_customer"].fillna(False).astype(int)
    + 0.08 * np.clip(
        events["historical_transactions"].fillna(0) / 20, 0, 1
    )
    + 0.08 * np.clip(
        events["purchase_frequency_per_30d"].fillna(0) / 5, 0, 1
    )
    + 0.05 * np.clip(
        events["session_duration_minutes"].fillna(0) / 60, 0, 1
    )
)

risk_penalty = (
    0.10 * events["unusual_amount_flag"].fillna(False).astype(int)
    + 0.10 * events["unusual_location_flag"].fillna(False).astype(int)
    + 0.12 * events["velocity_flag"].fillna(False).astype(int)
    + 0.12 * events["previous_fraud_flag"].fillna(False).astype(int)
    + 0.06 * events["high_risk_device_flag"].fillna(False).astype(int)
)

p_retry = customer_strength.copy()
p_link = customer_strength * 0.85
p_reminder = customer_strength * 0.65
p_escalation = (
    0.35
    + 0.20 *
    events["historical_transactions"].fillna(0).clip(0, 20) / 20
)

p_retry += np.where(
    events["failure_reason"].isin(
        ["bank_timeout", "temporary_bank_error"]
    ), 0.22, 0
)

p_retry -= np.where(events["retry_count"] >= 2, 0.35, 0)

p_link += np.where(
    events["revenue_event"].eq("checkout_abandonment"),
    0.18, 0
)

p_reminder += np.where(
    events["revenue_event"].isin(
        ["invoice_overdue", "subscription_failure"]
    ), 0.18, 0
)

p_escalation += np.where(events["high_risk_case"], 0.35, 0)

p_retry -= risk_penalty
p_link -= risk_penalty * 0.75
p_reminder -= risk_penalty * 0.50

p_retry = np.clip(p_retry, 0.02, 0.95)
p_link = np.clip(p_link, 0.02, 0.95)
p_reminder = np.clip(p_reminder, 0.02, 0.95)
p_escalation = np.clip(p_escalation, 0.02, 0.95)

events["retry_expected_probability"] = p_retry.round(4)
events["payment_link_expected_probability"] = p_link.round(4)
events["reminder_expected_probability"] = p_reminder.round(4)
events["escalation_expected_probability"] = p_escalation.round(4)
''',

r'''# 9. Counterfactual expected recovery values

amount = events["transaction_amount"].clip(lower=1)

events["retry_expected_recovery"] = (
    amount * events["retry_expected_probability"]
).round(2)

events["payment_link_expected_recovery"] = (
    amount * events["payment_link_expected_probability"]
).round(2)

events["reminder_expected_recovery"] = (
    amount * events["reminder_expected_probability"]
).round(2)

events["escalation_expected_recovery"] = (
    amount * events["escalation_expected_probability"]
).round(2)

action_recovery_cols = {
    "RETRY_PAYMENT": "retry_expected_recovery",
    "CREATE_PAYMENT_LINK": "payment_link_expected_recovery",
    "SEND_REMINDER": "reminder_expected_recovery",
    "ESCALATE_TO_HUMAN": "escalation_expected_recovery"
}

matrix = events[list(action_recovery_cols.values())].to_numpy()

best_idx = matrix.argmax(axis=1)
action_names = list(action_recovery_cols.keys())

events["counterfactual_optimal_action_unconstrained"] = [
    action_names[i] for i in best_idx
]

events["estimated_optimal_recovery"] = matrix.max(axis=1).round(2)
''',

r'''# 10. Apply bounded recovery policy

def constrained_action(row):
    # Risk always overrides revenue optimization.
    if row["high_risk_case"]:
        return "ESCALATE_TO_HUMAN"

    # Never retry after two or more retries.
    if row["retry_count"] >= 2:
        allowed = {
            "CREATE_PAYMENT_LINK": row["payment_link_expected_recovery"],
            "SEND_REMINDER": row["reminder_expected_recovery"],
            "ESCALATE_TO_HUMAN": row["escalation_expected_recovery"]
        }
        return max(allowed, key=allowed.get)

    return row["counterfactual_optimal_action_unconstrained"]

events["policy_recommended_action"] = events.apply(
    constrained_action, axis=1
)

def selected_probability(row):
    return {
        "RETRY_PAYMENT": row["retry_expected_probability"],
        "CREATE_PAYMENT_LINK": row["payment_link_expected_probability"],
        "SEND_REMINDER": row["reminder_expected_probability"],
        "ESCALATE_TO_HUMAN": row["escalation_expected_probability"]
    }[row["policy_recommended_action"]]

events["selected_action_probability"] = events.apply(
    selected_probability, axis=1
).round(4)
''',

r'''# 11. Generate synthetic observed outcomes
#
# These are benchmark simulation labels, NOT real payment outcomes.
# They allow us to test the decision engine consistently.

events["actual_recovery_success"] = (
    rng.random(N_EVENTS)
    < events["selected_action_probability"].to_numpy()
)

events["actual_recovered_amount"] = np.where(
    events["actual_recovery_success"],
    events["transaction_amount"],
    0
).round(2)

events["actual_outcome"] = np.where(
    events["actual_recovery_success"],
    "recovered",
    np.where(
        events["policy_recommended_action"].eq("ESCALATE_TO_HUMAN"),
        "escalated",
        "not_recovered"
    )
)
''',

r'''# 12. Fixed-rule baseline

def baseline_action(row):
    if row["high_risk_case"]:
        return "ESCALATE_TO_HUMAN"

    if row["revenue_event"] == "payment_failure":
        return (
            "RETRY_PAYMENT"
            if row["retry_count"] < 2
            else "SEND_REMINDER"
        )

    if row["revenue_event"] == "checkout_abandonment":
        return "CREATE_PAYMENT_LINK"

    if row["revenue_event"] in [
        "subscription_failure", "invoice_overdue"
    ]:
        return "SEND_REMINDER"

    if row["revenue_event"] == "repeated_payment_failure":
        return "ESCALATE_TO_HUMAN"

    return "SEND_REMINDER"

events["baseline_action"] = events.apply(
    baseline_action, axis=1
)

probability_column = {
    "RETRY_PAYMENT": "retry_expected_probability",
    "CREATE_PAYMENT_LINK": "payment_link_expected_probability",
    "SEND_REMINDER": "reminder_expected_probability",
    "ESCALATE_TO_HUMAN": "escalation_expected_probability"
}

events["baseline_expected_recovery"] = events.apply(
    lambda row:
        row["transaction_amount"]
        * row[probability_column[row["baseline_action"]]],
    axis=1
).round(2)

events["baseline_recovery_success"] = (
    rng.random(N_EVENTS)
    < events.apply(
        lambda row:
            row[probability_column[row["baseline_action"]]],
        axis=1
    ).to_numpy()
)

events["baseline_actual_recovered_amount"] = np.where(
    events["baseline_recovery_success"],
    events["transaction_amount"],
    0
).round(2)
''',

r'''# 13. Select only useful final columns

final_columns = [
    "case_id",

    # Revenue event
    "transaction_amount",
    "payment_method",
    "device_type",
    "revenue_event",
    "failure_reason",
    "recovery_eligible",
    "retry_count",

    # Customer behavior
    "customer_age",
    "customer_gender",
    "customer_city",
    "session_duration_minutes",
    "pages_viewed",
    "is_returning_customer",

    # Long-term customer intelligence
    "historical_transactions",
    "historical_total_spend",
    "historical_average_order_value",
    "historical_max_order_value",
    "purchase_frequency_per_30d",
    "days_since_last_purchase",
    "distinct_products",
    "customer_value_segment",
    "favorite_product",

    # Risk signals
    "failed_transactions_24h",
    "transaction_frequency_24h",
    "unusual_amount_flag",
    "unusual_location_flag",
    "multiple_transactions_short_time",
    "high_risk_device_flag",
    "velocity_flag",
    "previous_fraud_flag",
    "risk_signal_count",
    "automated_recovery_risk",

    # Counterfactual benchmark labels
    "retry_expected_probability",
    "payment_link_expected_probability",
    "reminder_expected_probability",
    "escalation_expected_probability",
    "retry_expected_recovery",
    "payment_link_expected_recovery",
    "reminder_expected_recovery",
    "escalation_expected_recovery",
    "counterfactual_optimal_action_unconstrained",
    "estimated_optimal_recovery",

    # Policy + outcome
    "policy_recommended_action",
    "selected_action_probability",
    "actual_outcome",
    "actual_recovery_success",
    "actual_recovered_amount",

    # Baseline
    "baseline_action",
    "baseline_expected_recovery",
    "baseline_recovery_success",
    "baseline_actual_recovered_amount"
]

final_df = events[final_columns].copy()

# Clean impossible/infinite values
final_df = final_df.replace([np.inf, -np.inf], np.nan)
final_df = final_df[
    final_df["transaction_amount"].notna()
    & (final_df["transaction_amount"] > 0)
].reset_index(drop=True)

print(f"Final benchmark: {len(final_df):,} rows × {len(final_df.columns)} columns")
''',

r'''# 14. Save benchmark + train/validation/test

output_dir = Path.cwd() / "revenue_recovery_output"
output_dir.mkdir(parents=True, exist_ok=True)

benchmark_path = output_dir / "revenue_recovery_benchmark.csv"
final_df.to_csv(benchmark_path, index=False)

# Reproducible split
split_rng = np.random.default_rng(123)
indices = np.arange(len(final_df))
split_rng.shuffle(indices)

train_end = int(0.70 * len(indices))
val_end = int(0.85 * len(indices))

train_df = final_df.iloc[indices[:train_end]].reset_index(drop=True)
validation_df = final_df.iloc[indices[train_end:val_end]].reset_index(drop=True)
test_df = final_df.iloc[indices[val_end:]].reset_index(drop=True)

train_path = output_dir / "revenue_recovery_train.csv"
val_path = output_dir / "revenue_recovery_validation.csv"
test_path = output_dir / "revenue_recovery_test.csv"

train_df.to_csv(train_path, index=False)
validation_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("Files created:")
print(benchmark_path.resolve())
print(train_path.resolve())
print(val_path.resolve())
print(test_path.resolve())
''',

r'''# 15. Quality report

total_at_risk = final_df["transaction_amount"].sum()
actual_recovered = final_df["actual_recovered_amount"].sum()
baseline_recovered = final_df["baseline_actual_recovered_amount"].sum()

print("=" * 65)
print("REVENUE COUNTERFACTUAL DATASET — QUALITY REPORT")
print("=" * 65)

print(f"Rows:                 {len(final_df):,}")
print(f"Columns:              {len(final_df.columns)}")
print(f"Revenue at risk:      ₹{total_at_risk:,.2f}")
print(f"Agent recovered:      ₹{actual_recovered:,.2f}")
print(f"Baseline recovered:   ₹{baseline_recovered:,.2f}")

if total_at_risk:
    print(
        f"Agent recovery rate:  "
        f"{100 * actual_recovered / total_at_risk:.2f}%"
    )

print("\nEvent distribution:")
print(final_df["revenue_event"].value_counts())

print("\nPolicy action distribution:")
print(final_df["policy_recommended_action"].value_counts())

print("\nRisk distribution:")
print(final_df["automated_recovery_risk"].value_counts())

print("\nMissing values (top 15):")
missing = final_df.isna().sum()
print(missing[missing > 0].sort_values(ascending=False).head(15))

print("\nSplit sizes:")
print(f"Train:      {len(train_df):,}")
print(f"Validation: {len(validation_df):,}")
print(f"Test:       {len(test_df):,}")

print("\nIMPORTANT:")
print("Counterfactual probabilities and outcomes are synthetic benchmark labels.")
print("Do not present them as real-world observed payment recoveries.")
''',

r'''# 16. Preview the final dataset

try:
    from IPython.display import display
    display(final_df.head(10))
except ImportError:
    print(final_df.head(10).to_string())

print("\nFinal column list:")
for c in final_df.columns:
    print(" -", c)
'''
]

nb = {
    "cells": [
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": src.splitlines(True)
        }
        for src in cells_source
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

output = Path("/Users/kesavasushanth/Downloads/Razorpay-training-dataset/training-dataset.ipynb")
output.write_text(json.dumps(nb, indent=2), encoding="utf-8")

print(f"Created clean notebook: {output}")


Created clean notebook: /Users/kesavasushanth/Downloads/Razorpay-training-dataset/training-dataset.ipynb
